# RQ3 — FDA (échange d'amplitude basse fréquence)

Run `unet3d_fda_pli0` du tableau d'ablation : UNet MONAI 3D en fp32 (clipping des gradients à 12), patches de 96³, **pli 0** de `splits.json`, **graine 2026**,
40 000 itérations (lots de 4 patches), identiques pour toutes les techniques. Durée estimée sur une T4 : 7 à 9 h
(garde-fou à 10 h d'entraînement, session limitée à 12 h).

**Avant de lancer** : accélérateur *GPU T4*, Internet activé (installation de MONAI et TorchIO), datasets privés
`atlas-c1-processed` (données C1) et `projet-avc-code` (code) ajoutés en entrée.
Notebook à garder **privé** (accord d'utilisation ATLAS). *Save Version → Save & Run All* pour une exécution en arrière-plan.

In [ ]:
# Code du projet (dataset privé projet-avc-code) et dépendances absentes de l'image Kaggle.
import hashlib, json, shutil, subprocess, sys, zipfile
from pathlib import Path

def find_input(slug):  # dossier d'un dataset attaché, quel que soit le chemin de montage de Kaggle
    for pattern in (slug, f"*/{slug}", f"*/*/{slug}", f"*/*/*/{slug}"):
        found = sorted(Path("/kaggle/input").glob(pattern))
        if found:
            return found[0]
    raise FileNotFoundError(f"Dataset {slug} non attaché : " + str(sorted(Path("/kaggle/input").glob("*"))))

CODE = find_input("projet-avc-code")
REPO = Path("/kaggle/working/PROJET_AVC")
if not REPO.exists():
    archives = sorted(CODE.rglob("projet_avc_code.zip"))
    if archives:
        zipfile.ZipFile(archives[0]).extractall(REPO)
    else:  # Kaggle a décompressé l'archive : copie de l'arborescence
        shutil.copytree(next(CODE.rglob("adaptation/train.py")).parents[1], REPO)
print(json.loads((REPO / "VERSION.json").read_text()))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "monai==1.6.1", "torchio==1.2.1", "nibabel==5.4.2",
                "pyarrow"], check=True)
import torch; print(torch.__version__, torch.cuda.get_device_name(0))

In [ ]:
# Données C1 : liens vers les dossiers de sujets, quelle que soit l'archive d'origine.
INPUT = find_input("atlas-c1-processed")
DATA = Path("/kaggle/working/processed")
DATA.mkdir(exist_ok=True)
sources = [INPUT]
if not any(INPUT.rglob("qc.json")):  # archives restées compressées : extraction sur le disque de travail
    extracted = Path("/kaggle/working/c1_extrait")
    for archive in sorted(INPUT.rglob("processed_partie*.zip")):
        zipfile.ZipFile(archive).extractall(extracted)
    sources = [extracted]
for qc in (path for source in sources for path in source.rglob("qc.json")):
    link = DATA / qc.parent.name
    if not link.exists():
        link.symlink_to(qc.parent, target_is_directory=True)
table = DATA / "subjects.parquet"
if not table.exists():
    table.symlink_to(next(INPUT.rglob("subjects.parquet")))
print(sum(1 for _ in DATA.glob("*/qc.json")), "sujets prétraités")
# Même découpage que le code : splits.json du dataset identique à celui du dépôt.
digest = lambda path: hashlib.sha256(Path(path).read_bytes()).hexdigest()
assert digest(next(INPUT.rglob("splits.json"))) == digest(REPO / "splits.json"), "splits.json différent du dépôt"

In [ ]:
# Entraînement, validation sur le pli, puis évaluation sur le test ID du pli et le test OOD.
# Session interrompue : rattacher la sortie précédente, copier last.pt dans /kaggle/working/runs/unet3d_fda_pli0/ et ajouter --resume.
!cd {REPO} && python -u -m adaptation.train --technique fda --fold 0 --seed 2026 --data-root {DATA} --output /kaggle/working/runs --num-workers 4 --validate-every 2000 --log-every 100

In [ ]:
# Résumé et fichiers à récupérer (onglet Output) ; aucune image de patient n'est écrite.
for path in sorted(Path("/kaggle/working/runs/unet3d_fda_pli0").glob("resume*.json")):
    print(path.name, json.dumps(json.loads(path.read_text()), indent=2, ensure_ascii=False))
!ls -la /kaggle/working/runs/unet3d_fda_pli0